In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

import numpy as np

features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

from training_info import TrainingInfo

TrainingInfo.update_metadata()
metadata = TrainingInfo.load_metadata()

2025-09-24 16:22:53,174 - INFO - Adding 0 files, removing 0 hashes
0it [00:00, ?it/s]


In [2]:
import pandas as pd

experiment_name = "base_fvt_training_ensemble_ZH4b"
hashes = TrainingInfo.find({"experiment_name": experiment_name})

hparams_df = pd.DataFrame([
    {"hash": hash_, 
     "seed": metadata[hash_]["dataset"]["seed"], 
     "signal_ratio": metadata[hash_]["dataset"]["signal_ratio"], 
     "n_3b": metadata[hash_]["dataset"]["n_3b"], 
     "ratio_4b": metadata[hash_]["dataset"]["ratio_4b"], 
     "signal_filename": metadata[hash_]["dataset"]["signal_filename"], 
     "train_seed": metadata[hash_]["train_seed"],
    }
    for hash_ in hashes
])

In [3]:
from training_info import TrainingInfo
import pandas as pd

experiment_name = "smeared_fvt_training_ensemble_ZH4b"
TrainingInfo.update_metadata()
metadata = TrainingInfo.load_metadata()

hashes = TrainingInfo.find({"experiment_name": experiment_name})

hparams_df = pd.DataFrame([
    {"hash": hash_, 
     "seed": metadata[hash_]["dataset"]["seed"], 
     "signal_ratio": metadata[hash_]["dataset"]["signal_ratio"], 
     "n_3b": metadata[hash_]["dataset"]["n_3b"], 
     "ratio_4b": metadata[hash_]["dataset"]["ratio_4b"], 
     "signal_filename": metadata[hash_]["dataset"]["signal_filename"], 
     "train_seed": metadata[hash_]["train_seed"],
     "noise_scale": metadata[hash_]["smearing"]["noise_scale"],
    }
    for hash_ in hashes
])

2025-09-24 16:23:19,793 - INFO - Adding 0 files, removing 0 hashes
0it [00:00, ?it/s]


In [4]:
counts = (hparams_df.groupby(["train_seed", "signal_ratio", "noise_scale"])["hash"].count() != 100)
corrupted_hyperparams = counts[counts].reset_index()[["train_seed", "signal_ratio", "noise_scale"]].values

corrupted_hashes = []
for train_seed, signal_ratio, noise_scale in corrupted_hyperparams:
    new_hashes = TrainingInfo.find({
        "experiment_name": "smeared_fvt_training_ensemble_ZH4b", 
        "train_seed": train_seed,
        "dataset": lambda x: (x["signal_ratio"] == signal_ratio),
        "smearing": lambda x: (x["noise_scale"] == noise_scale)
    })
    print(len(new_hashes))
    corrupted_hashes.extend(new_hashes)

len(corrupted_hashes)

99
90
1


190

In [12]:
hparams_df[["signal_ratio", "noise_scale", "train_seed"]].drop_duplicates()

u_signal_ratios = hparams_df["signal_ratio"].unique()
u_noise_scales = hparams_df["noise_scale"].unique()
u_train_seeds = hparams_df["train_seed"].unique()

from itertools import product

{(s, n, t) for s, n, t in product(u_signal_ratios, u_noise_scales, u_train_seeds)} - set(
    tuple(x) for x in hparams_df[["signal_ratio", "noise_scale", "train_seed"]].values)

{(0.02, 0.5, 8)}

In [4]:
from training_info import TrainingInfo
import pandas as pd

experiment_name = "CR_fvt_training_ensemble_max_ZH4b"
TrainingInfo.update_metadata()
metadata = TrainingInfo.load_metadata()

hashes = TrainingInfo.find({"experiment_name": experiment_name})
len(hashes)

2025-09-07 22:49:39,703 - INFO - Adding 62 files, removing 0 hashes
100%|██████████| 62/62 [00:00<00:00, 484.25it/s]


6817

In [13]:
from training_info import TrainingInfo
import pandas as pd

experiment_name = "CR_fvt_training_ensemble_max_ZH4b"
TrainingInfo.update_metadata()
metadata = TrainingInfo.load_metadata()

hashes = TrainingInfo.find({"experiment_name": experiment_name})

hparams_df = pd.DataFrame([
    {"hash": hash_, 
     "seed": metadata[hash_]["dataset"]["seed"], 
     "signal_ratio": metadata[hash_]["dataset"]["signal_ratio"], 
     "n_3b": metadata[hash_]["dataset"]["n_3b"], 
     "ratio_4b": metadata[hash_]["dataset"]["ratio_4b"], 
     "signal_filename": metadata[hash_]["dataset"]["signal_filename"], 
     "train_seed": metadata[hash_]["train_seed"],
     "noise_scale": metadata[metadata[hash_]["signal_region"]["SR_stats_hashes"][0]]["smearing"]["noise_scale"],
     "SR_size": metadata[hash_]["signal_region"]["4b_in_SR"],
    }
    for hash_ in hashes
])

2025-09-07 12:58:49,195 - INFO - Adding 0 files, removing 0 hashes
0it [00:00, ?it/s]


array([[0.    , 0.005 , 1.    , 0.05  ],
       [0.    , 0.005 , 1.    , 0.1   ],
       [0.    , 0.005 , 1.    , 0.15  ],
       [0.    , 0.0075, 1.    , 0.05  ],
       [0.    , 0.0075, 1.    , 0.1   ],
       [0.    , 0.0075, 1.    , 0.15  ],
       [0.    , 0.0075, 2.    , 0.2   ],
       [0.    , 0.01  , 1.    , 0.05  ],
       [0.    , 0.01  , 2.    , 0.2   ],
       [0.    , 0.02  , 1.    , 0.05  ],
       [0.    , 0.02  , 1.    , 0.1   ],
       [0.    , 0.02  , 1.    , 0.15  ],
       [0.    , 0.02  , 2.    , 0.2   ]])

In [5]:
TrainingInfo.delete(corrupted_hashes)

Deleting 102 hashes


2025-09-02 23:08:18,790 - INFO - Adding 0 files, removing 102 hashes
0it [00:00, ?it/s]


In [ ]:
from training_info import TrainingInfo
import pandas as pd

experiment_name = "CR_fvt_training_ensemble_max"
TrainingInfo.update_metadata()
metadata = TrainingInfo.load_metadata()

hashes = TrainingInfo.find({
    "experiment_name": experiment_name})

print(len(hashes))

: 

In [5]:
from training_info import TrainingInfo
import pandas as pd

experiment_name = "smeared_fvt_training_ensemble"
TrainingInfo.update_metadata()
metadata = TrainingInfo.load_metadata()

hashes = TrainingInfo.find({
    "experiment_name": experiment_name, 
    "dataset": lambda x: x["seed"] >= 50
                   })


hparams_df = pd.DataFrame([
    {"hash": hash_, 
     "seed": metadata[hash_]["dataset"]["seed"], 
     "signal_ratio": metadata[hash_]["dataset"]["signal_ratio"], 
     "n_3b": metadata[hash_]["dataset"]["n_3b"], 
     "ratio_4b": metadata[hash_]["dataset"]["ratio_4b"], 
     "signal_filename": metadata[hash_]["dataset"]["signal_filename"], 
     "train_seed": metadata[hash_]["train_seed"],
     "noise_scale": metadata[hash_]["smearing"]["noise_scale"],
    }
    for hash_ in hashes
])

2025-08-27 14:23:50,859 - INFO - Adding 700 files, removing 0 hashes
100%|██████████| 700/700 [00:06<00:00, 111.97it/s]


In [5]:
hashes = TrainingInfo.find({
        "experiment_name": "CR_fvt_training_ensemble_max", 
        "dataset": lambda x: (x["seed"] >= 50),
    })
len(hashes)

5000

In [ ]:
counts = (hparams_df.groupby(["train_seed", "signal_ratio", "noise_scale"])["hash"].count() != 50)
corrupted_hyperparams = counts[counts].reset_index()[["train_seed", "signal_ratio", "noise_scale"]].values

corrupted_hashes = []
for train_seed, signal_ratio, noise_scale in corrupted_hyperparams:
    new_hashes = TrainingInfo.find({
        "experiment_name": "smeared_fvt_training_ensemble_ZH4b", 
        "train_seed": train_seed,
        "dataset": lambda x: (x["seed"] >= 50)
                              and (x["signal_ratio"] == signal_ratio),
        "smearing": lambda x: (x["noise_scale"] == noise_scale)
    })
    print(len(new_hashes))
    corrupted_hashes.extend(new_hashes)

len(corrupted_hashes)

0

In [ ]:
counts = (hparams_df.groupby(["train_seed", "signal_ratio", "noise_scale", "SR_size"])["hash"].count() != 100)
corrupted_hyperparams = counts[counts].reset_index()[["train_seed", "signal_ratio", "noise_scale", "SR_size"]].values
corrupted_hyperparams
corrupted_hashes = []
for train_seed, signal_ratio, noise_scale, SR_size in corrupted_hyperparams:
    new_hashes = TrainingInfo.find({
        "experiment_name": "CR_fvt_training_ensemble_max_ZH4b", 
        "train_seed": train_seed,
        "dataset": lambda x: x["signal_ratio"] == signal_ratio,
        "smearing": lambda x: (x["noise_scale"] == noise_scale)
    })
    print(len(new_hashes))
    corrupted_hashes.extend(new_hashes)

len(corrupted_hashes)

In [ ]:
# counts = (hparams_df.groupby(["train_seed", "signal_ratio", "noise_scale"])["hash"].count() != 50)
# corrupted_hyperparams = counts[counts].reset_index()[["train_seed", "signal_ratio", "noise_scale"]].values

# corrupted_hashes = []
# for train_seed, signal_ratio, noise_scale in corrupted_hyperparams:
#     new_hashes = TrainingInfo.find({
#         "experiment_name": "smeared_fvt_training_ensemble", 
#         "train_seed": train_seed,
#         "dataset": lambda x: (x["seed"] >= 50)
#                               and (x["signal_ratio"] == signal_ratio),
#         "smearing": lambda x: (x["noise_scale"] == noise_scale)
#     })
#     print(len(new_hashes))
#     corrupted_hashes.extend(new_hashes)

# len(corrupted_hashes)

# TrainingInfo.delete(corrupted_hashes)

33
25
Deleting 58 hashes


2025-08-22 00:06:58,688 - INFO - Adding 0 files, removing 58 hashes
0it [00:00, ?it/s]


In [18]:
dupl = hparams_df.set_index(["seed", "train_seed", "signal_ratio", "noise_scale"])
duplicated_hashes = dupl.loc[dupl.index.duplicated(), "hash"].values

In [19]:
len(duplicated_hashes)

0

In [ ]:
# TrainingInfo.delete(duplicated_hashes)

Deleting 50 hashes


2025-08-22 00:00:53,179 - INFO - Adding 0 files, removing 50 hashes
0it [00:00, ?it/s]


In [ ]:
# # fetch all filenames in data/checkpoints/

# import glob

# filenames = glob.glob("/home/export/soheuny/SRFinder/soheun/data/checkpoints/*")

100925